# Rental Listing Validation, Feature Selection and Hyperparameter Optimization

**Dataset source:** Kaggle `Two Sigma Connect: Rental Listing Inquiries`


## 1. Environment Setup

In [3]:
import os
import time
import random
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    KFold as SklearnKFold,
    GroupKFold as SklearnGroupKFold,
    StratifiedKFold as SklearnStratifiedKFold,
    TimeSeriesSplit as SklearnTimeSeriesSplit,
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except Exception:
    root_mean_squared_error = None
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import chi2
from sklearn.base import clone

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

## 2. Data Loading

In [4]:
def find_data_file(filename: str) -> Path:
    possible_paths = [
        Path("data") / filename,
    ]

    for path in possible_paths:
        if path.exists():
            return path

    raise FileNotFoundError(f"Cannot find {filename}. Tried: {possible_paths}")


train_path = find_data_file("train.json")
test_path = find_data_file("test.json")

train_raw = pd.read_json(train_path)
test_raw = pd.read_json(test_path)

print("train:", train_raw.shape)
print("test :", test_raw.shape)

display(train_raw.head())
display(test_raw.head())

train: (49352, 15)
test : (74659, 14)


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address
0,1.0,1,79780be1514f645d7e6be99a3de696c5,2016-06-11 05:29:41,Large with awesome terrace--accessible via bed...,Suffolk Street,"[Elevator, Laundry in Building, Laundry in Uni...",40.7185,7142618,-73.9865,b1b1852c416d78d7765d746cb1b8921f,[https://photos.renthop.com/2/7142618_1c45a2c8...,2950,99 Suffolk Street
1,1.0,2,0,2016-06-24 06:36:34,Prime Soho - between Bleecker and Houston - Ne...,Thompson Street,"[Pre-War, Dogs Allowed, Cats Allowed]",40.7278,7210040,-74.0000,d0b5648017832b2427eeb9956d966a14,[https://photos.renthop.com/2/7210040_d824cc71...,2850,176 Thompson Street
2,1.0,0,0,2016-06-17 01:23:39,Spacious studio in Prime Location. Cleanbuildi...,Sullivan Street,"[Pre-War, Dogs Allowed, Cats Allowed]",40.7260,7174566,-74.0026,e6472c7237327dd3903b3d6f6a94515a,[https://photos.renthop.com/2/7174566_ba3a35c5...,2295,115 Sullivan Street
3,1.0,2,f9c826104b91d868e69bd25746448c0c,2016-06-21 05:06:02,For immediate access call Bryan.<br /><br />Bo...,Jones Street,"[Hardwood Floors, Dogs Allowed, Cats Allowed]",40.7321,7191391,-74.0028,41735645e0f8f13993c42894023f8e58,[https://photos.renthop.com/2/7191391_8c2f2d49...,2900,23 Jones Street
5,1.0,1,81062936e12ee5fa6cd2b965698e17d5,2016-06-16 07:24:27,Beautiful TRUE 1 bedroom in a luxury building ...,Exchange Place,"[Roof Deck, Doorman, Elevator, Fitness Center,...",40.7054,7171695,-74.0095,a742cf7dd3b2627d83417bc3a1b3ec96,[https://photos.renthop.com/2/7171695_089ffee2...,3254,20 Exchange Place


## 3. Preprocessing and Feature Engineering

Mandatory features to create:

`Elevator`, `HardwoodFloors`, `CatsAllowed`, `DogsAllowed`, `Doorman`, `Dishwasher`, `NoFee`, `LaundryinBuilding`, `FitnessCenter`, `Pre-War`, `LaundryinUnit`, `RoofDeck`, `OutdoorSpace`, `DiningRoom`, `HighSpeedInternet`, `Balcony`, `SwimmingPool`, `LaundryInBuilding`, `NewConstruction`, `Terrace`.

In [5]:
MANDATORY_AMENITIES = [
    "Elevator",
    "HardwoodFloors",
    "CatsAllowed",
    "DogsAllowed",
    "Doorman",
    "Dishwasher",
    "NoFee",
    "LaundryinBuilding",
    "FitnessCenter",
    "Pre-War",
    "LaundryinUnit",
    "RoofDeck",
    "OutdoorSpace",
    "DiningRoom",
    "HighSpeedInternet",
    "Balcony",
    "SwimmingPool",
    "LaundryInBuilding",
    "NewConstruction",
    "Terrace",
]

def normalize_feature_text(text) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()
    for ch in [" ", "_", "-", "/", "\\", ".", ",", "&"]:
        text = text.replace(ch, "")
    return text

def list_to_normalized_feature_set(features):
    if not isinstance(features, list):
        return set()
    return {normalize_feature_text(x) for x in features}

def preprocess_renthop(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    out = df.copy()

    out["created"] = pd.to_datetime(out["created"])
    out["created_timestamp"] = out["created"].astype("int64") // 10**9
    out["created_month"] = out["created"].dt.month
    out["created_day"] = out["created"].dt.day
    out["created_dayofweek"] = out["created"].dt.dayofweek
    out["created_hour"] = out["created"].dt.hour

    out["num_photos"] = out["photos"].apply(lambda x: len(x) if isinstance(x, list) else 0)
    out["num_features"] = out["features"].apply(lambda x: len(x) if isinstance(x, list) else 0)
    out["description_length"] = out["description"].fillna("").astype(str).str.len()
    out["description_word_count"] = out["description"].fillna("").astype(str).str.split().apply(len)

    out["rooms"] = out["bedrooms"] + out["bathrooms"]
    out["price_per_bedroom"] = out["price"] / (out["bedrooms"] + 1)
    out["price_per_bathroom"] = out["price"] / (out["bathrooms"] + 1)
    out["price_per_room"] = out["price"] / (out["rooms"] + 1)

    normalized_feature_sets = out["features"].apply(list_to_normalized_feature_set)

    for amenity in MANDATORY_AMENITIES:
        normalized_amenity = normalize_feature_text(amenity)
        out[amenity] = normalized_feature_sets.apply(lambda s: int(normalized_amenity in s))

    if is_train and "interest_level" in out.columns:
        target_map = {"low": 0, "medium": 1, "high": 2}
        out["target"] = out["interest_level"].map(target_map).astype(int)
        out["target_class"] = out["target"]

    return out

train = preprocess_renthop(train_raw, is_train=True)
test = preprocess_renthop(test_raw, is_train=False)

display(train[MANDATORY_AMENITIES + ["interest_level", "target"]].head())
print(train[MANDATORY_AMENITIES].sum().sort_values(ascending=False))

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,interest_level,target
4,0,1,1,1,0,1,0,1,0,1,...,0,1,0,0,0,1,0,0,medium,1
6,1,1,0,0,1,1,1,1,0,0,...,0,0,0,0,0,1,0,0,low,0
9,1,1,0,0,1,1,0,1,0,0,...,0,0,0,0,0,1,0,0,medium,1
10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,medium,1
15,1,0,0,0,1,0,0,1,1,0,...,0,0,0,0,0,1,0,0,low,0


Elevator             26272
HardwoodFloors       23559
CatsAllowed          23540
DogsAllowed          22035
Doorman              20966
Dishwasher           20806
LaundryinBuilding    18638
LaundryInBuilding    18638
NoFee                18065
FitnessCenter        13252
Pre-War              10501
LaundryinUnit         9345
RoofDeck              7010
OutdoorSpace          5270
DiningRoom            5150
HighSpeedInternet     4315
Balcony               3058
SwimmingPool          2730
NewConstruction       2608
Terrace               2313
dtype: int64


## 4. Feature Matrix

In [6]:
BASE_NUMERIC_FEATURES = [
    "bathrooms",
    "bedrooms",
    "latitude",
    "longitude",
    "price",
    "created_timestamp",
    "created_month",
    "created_day",
    "created_dayofweek",
    "created_hour",
    "num_photos",
    "num_features",
    "description_length",
    "description_word_count",
    "rooms",
    "price_per_bedroom",
    "price_per_bathroom",
    "price_per_room",
]

FEATURE_COLUMNS = BASE_NUMERIC_FEATURES + MANDATORY_AMENITIES

X = train[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
y = train["target"]

X = X.fillna(X.median(numeric_only=True))

print("X shape:", X.shape)
print("y distribution:")
display(y.value_counts(normalize=True).sort_index())
display(X.head())

X shape: (49352, 38)
y distribution:


target
0    0.694683
1    0.227529
2    0.077788
Name: proportion, dtype: float64

,bathrooms,bedrooms,latitude,longitude,price,created_timestamp,created_month,created_day,created_dayofweek,created_hour,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,1.0,1,40.7108,-73.9539,2400,1466056527,6,16,3,5,...,0,0,0,1,0,0,0,1,0,0
6,1.0,2,40.7513,-73.9722,3800,1464759873,6,1,2,5,...,0,0,0,0,0,0,0,1,0,0
9,1.0,2,40.7575,-73.9625,3495,1465917599,6,14,1,15,...,1,0,0,0,0,0,0,1,0,0
10,1.5,3,40.7145,-73.9425,3000,1466754864,6,24,4,7,...,0,0,0,0,0,0,0,0,0,0
15,1.0,0,40.7439,-73.9743,2795,1467085823,6,28,1,3,...,0,0,0,0,0,0,0,1,0,0


## 5. Quality Metrics

Because Lasso and ElasticNet are regression models, the ordinal target is treated as a regression target:

- low = 0
- medium = 1
- high = 2

We measure:

- MAE;
- RMSE;
- R2.

In [7]:
def rmse_score(y_true, y_pred) -> float:
    if root_mean_squared_error is not None:
        return root_mean_squared_error(y_true, y_pred)
    return np.sqrt(mean_squared_error(y_true, y_pred))

def regression_metrics(y_true, y_pred, prefix: str = "") -> dict:
    return {
        f"{prefix}MAE": mean_absolute_error(y_true, y_pred),
        f"{prefix}RMSE": rmse_score(y_true, y_pred),
        f"{prefix}R2": r2_score(y_true, y_pred),
    }

def fit_evaluate_model(model, X_train, y_train, X_valid, y_valid, model_name="model") -> dict:
    model = clone(model)
    start = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start

    pred_train = model.predict(X_train)
    pred_valid = model.predict(X_valid)

    result = {"model": model_name, "fit_time_sec": fit_time}
    result.update(regression_metrics(y_train, pred_train, prefix="train_"))
    result.update(regression_metrics(y_valid, pred_valid, prefix="valid_"))
    result["overfit_gap_RMSE"] = result["valid_RMSE"] - result["train_RMSE"]
    return result

## 6. Custom Validation Split Functions

### What does deterministic split mean?

A deterministic split gives the same train/test indices every time the code is executed with the same input data and the same random seed.This is important because:

- experiments become reproducible;
- metrics can be compared fairly;
- debugging becomes easier.

In [8]:
def random_train_test_split_custom(df: pd.DataFrame, test_size: float, random_state: int = 42):
    assert 0 < test_size < 1, "test_size must be between 0 and 1"
    rng = np.random.RandomState(random_state)
    indices = np.array(df.index)
    shuffled = indices.copy()
    rng.shuffle(shuffled)

    n_test = int(round(len(df) * test_size))
    test_idx = shuffled[:n_test]
    train_idx = shuffled[n_test:]

    return df.loc[train_idx].copy(), df.loc[test_idx].copy()

def random_train_valid_test_split_custom(
    df: pd.DataFrame,
    validation_size: float,
    test_size: float,
    random_state: int = 42,
):
    assert 0 < validation_size < 1
    assert 0 < test_size < 1
    assert validation_size + test_size < 1, "validation_size + test_size must be < 1"

    rng = np.random.RandomState(random_state)
    indices = np.array(df.index)
    shuffled = indices.copy()
    rng.shuffle(shuffled)

    n_test = int(round(len(df) * test_size))
    n_valid = int(round(len(df) * validation_size))

    test_idx = shuffled[:n_test]
    valid_idx = shuffled[n_test:n_test + n_valid]
    train_idx = shuffled[n_test + n_valid:]

    return df.loc[train_idx].copy(), df.loc[valid_idx].copy(), df.loc[test_idx].copy()

def date_train_test_split_custom(df: pd.DataFrame, date_field: str, date_split):
    date_split = pd.to_datetime(date_split)
    sorted_df = df.sort_values(date_field)
    train_part = sorted_df[sorted_df[date_field] < date_split].copy()
    test_part = sorted_df[sorted_df[date_field] >= date_split].copy()
    return train_part, test_part

def date_train_valid_test_split_custom(
    df: pd.DataFrame,
    date_field: str,
    validation_date,
    test_date,
):
    validation_date = pd.to_datetime(validation_date)
    test_date = pd.to_datetime(test_date)
    assert validation_date < test_date, "validation_date must be earlier than test_date"

    sorted_df = df.sort_values(date_field)
    train_part = sorted_df[sorted_df[date_field] < validation_date].copy()
    valid_part = sorted_df[
        (sorted_df[date_field] >= validation_date) & (sorted_df[date_field] < test_date)
    ].copy()
    test_part = sorted_df[sorted_df[date_field] >= test_date].copy()

    return train_part, valid_part, test_part

def date_train_valid_test_split_by_ratio(
    df: pd.DataFrame,
    date_field: str,
    train_size: float = 0.6,
    validation_size: float = 0.2,
    test_size: float = 0.2,
):
    assert abs(train_size + validation_size + test_size - 1.0) < 1e-9
    sorted_df = df.sort_values(date_field).copy()
    n = len(sorted_df)
    train_end = int(n * train_size)
    valid_end = int(n * (train_size + validation_size))

    train_part = sorted_df.iloc[:train_end].copy()
    valid_part = sorted_df.iloc[train_end:valid_end].copy()
    test_part = sorted_df.iloc[valid_end:].copy()

    return train_part, valid_part, test_part

### 6.1 Demonstration of custom one-fold splits

In [9]:
train_2, test_2 = random_train_test_split_custom(train, test_size=0.2, random_state=RANDOM_STATE)
train_3, valid_3, test_3 = random_train_valid_test_split_custom(
    train, validation_size=0.2, test_size=0.2, random_state=RANDOM_STATE
)

print("Random 2-part split:", train_2.shape, test_2.shape)
print("Random 3-part split:", train_3.shape, valid_3.shape, test_3.shape)

date_split = train["created"].quantile(0.8)
train_date_2, test_date_2 = date_train_test_split_custom(train, "created", date_split=date_split)
print("Date 2-part split:", train_date_2.shape, test_date_2.shape, "date_split:", date_split)

validation_date = train["created"].quantile(0.6)
test_date = train["created"].quantile(0.8)
train_date_3, valid_date_3, test_date_3 = date_train_valid_test_split_custom(
    train, "created", validation_date=validation_date, test_date=test_date
)
print("Date 3-part split:", train_date_3.shape, valid_date_3.shape, test_date_3.shape)
print("validation_date:", validation_date, "test_date:", test_date)

Random 2-part split: (39482, 50) (9870, 50)
Random 3-part split: (29612, 50) (9870, 50) (9870, 50)
Date 2-part split: (39481, 50) (9871, 50) date_split: 2016-06-12 08:18:36.600000
Date 3-part split: (29611, 50) (9870, 50) (9871, 50)
validation_date: 2016-05-25 01:11:19.200000 test_date: 2016-06-12 08:18:36.600000


## 7. Custom Cross-Validation Methods


In [10]:
def kfold_indices_custom(df: pd.DataFrame, k: int, random_state: int = 42, shuffle: bool = True):
    assert k >= 2
    indices = np.array(df.index)

    if shuffle:
        rng = np.random.RandomState(random_state)
        indices = indices.copy()
        rng.shuffle(indices)

    folds = np.array_split(indices, k)
    result = []

    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        result.append((train_idx, test_idx))

    return result

def grouped_kfold_indices_custom(df: pd.DataFrame, k: int, group_field: str):
    assert k >= 2
    group_sizes = df.groupby(group_field).size().sort_values(ascending=False)

    fold_groups = [set() for _ in range(k)]
    fold_sizes = np.zeros(k, dtype=int)

    # Assign larger groups to the currently smallest fold.
    for group_value, size in group_sizes.items():
        fold_id = int(np.argmin(fold_sizes))
        fold_groups[fold_id].add(group_value)
        fold_sizes[fold_id] += size

    result = []
    for i in range(k):
        test_mask = df[group_field].isin(fold_groups[i])
        test_idx = df.index[test_mask].to_numpy()
        train_idx = df.index[~test_mask].to_numpy()
        result.append((train_idx, test_idx))

    return result

def stratified_kfold_indices_custom(
    df: pd.DataFrame,
    k: int,
    stratify_field: str,
    random_state: int = 42,
):
    assert k >= 2
    rng = np.random.RandomState(random_state)

    fold_indices = [[] for _ in range(k)]

    for _, group in df.groupby(stratify_field):
        indices = group.index.to_numpy().copy()
        rng.shuffle(indices)
        parts = np.array_split(indices, k)
        for fold_id, part in enumerate(parts):
            fold_indices[fold_id].extend(part.tolist())

    all_indices = set(df.index)
    result = []
    for i in range(k):
        test_idx = np.array(fold_indices[i])
        train_idx = np.array(list(all_indices - set(test_idx)))
        result.append((train_idx, test_idx))

    return result

def time_series_split_indices_custom(df: pd.DataFrame, k: int, date_field: str):
    assert k >= 2
    sorted_indices = df.sort_values(date_field).index.to_numpy()
    folds = np.array_split(sorted_indices, k)

    result = []
    # We train k-1 models: first fold train, second fold test; then expanding window.
    for i in range(1, k):
        train_idx = np.concatenate(folds[:i])
        test_idx = folds[i]
        result.append((train_idx, test_idx))

    return result

### 7.1 Demonstration of custom cross-validation methods

In [11]:
K = 5

custom_kfold = kfold_indices_custom(train, k=K, random_state=RANDOM_STATE)
custom_group_kfold = grouped_kfold_indices_custom(train, k=K, group_field="manager_id")
custom_stratified_kfold = stratified_kfold_indices_custom(train, k=K, stratify_field="target_class", random_state=RANDOM_STATE)
custom_time_series = time_series_split_indices_custom(train, k=K, date_field="created")

for name, folds in [
    ("KFold", custom_kfold),
    ("GroupKFold", custom_group_kfold),
    ("StratifiedKFold", custom_stratified_kfold),
    ("TimeSeriesSplit", custom_time_series),
]:
    print(name, "number of splits:", len(folds))
    print("first split train/test:", len(folds[0][0]), len(folds[0][1]))

KFold number of splits: 5
first split train/test: 39481 9871
GroupKFold number of splits: 5
first split train/test: 39481 9871
StratifiedKFold number of splits: 5
first split train/test: 39481 9871
TimeSeriesSplit number of splits: 4
first split train/test: 9871 9871


## 8. Custom vs scikit-learn Validation Comparison

We compare feature distributions for the training part.

For numeric features, we compare:

- mean difference;
- standard deviation difference.

For target distribution, we compare class ratios.

In [12]:
def get_sklearn_kfold_indices(df, k=5):
    splitter = SklearnKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    return [(df.index[tr].to_numpy(), df.index[te].to_numpy()) for tr, te in splitter.split(df)]

def get_sklearn_group_kfold_indices(df, k=5, group_field="manager_id"):
    splitter = SklearnGroupKFold(n_splits=k)
    return [
        (df.index[tr].to_numpy(), df.index[te].to_numpy())
        for tr, te in splitter.split(df, groups=df[group_field])
    ]

def get_sklearn_stratified_kfold_indices(df, k=5, stratify_field="target_class"):
    splitter = SklearnStratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    return [
        (df.index[tr].to_numpy(), df.index[te].to_numpy())
        for tr, te in splitter.split(df, df[stratify_field])
    ]

def get_sklearn_time_series_split_indices(df, k=5, date_field="created"):
    sorted_df = df.sort_values(date_field)
    splitter = SklearnTimeSeriesSplit(n_splits=k-1)
    return [
        (sorted_df.index[tr].to_numpy(), sorted_df.index[te].to_numpy())
        for tr, te in splitter.split(sorted_df)
    ]

def compare_train_distributions(df, custom_folds, sklearn_folds, features, target_col="target"):
    rows = []

    for fold_id, ((custom_train_idx, _), (sk_train_idx, _)) in enumerate(zip(custom_folds, sklearn_folds), start=1):
        custom_train = df.loc[custom_train_idx]
        sk_train = df.loc[sk_train_idx]

        for feature in features:
            rows.append({
                "fold": fold_id,
                "feature": feature,
                "custom_mean": custom_train[feature].mean(),
                "sklearn_mean": sk_train[feature].mean(),
                "abs_mean_diff": abs(custom_train[feature].mean() - sk_train[feature].mean()),
                "custom_std": custom_train[feature].std(),
                "sklearn_std": sk_train[feature].std(),
                "abs_std_diff": abs(custom_train[feature].std() - sk_train[feature].std()),
            })

    return pd.DataFrame(rows)

comparison_features = [
    "bathrooms", "bedrooms", "price", "latitude", "longitude",
    "num_photos", "num_features", "description_word_count",
    "Elevator", "DogsAllowed", "CatsAllowed"
]

sk_kfold = get_sklearn_kfold_indices(train, k=K)
sk_group = get_sklearn_group_kfold_indices(train, k=K, group_field="manager_id")
sk_strat = get_sklearn_stratified_kfold_indices(train, k=K, stratify_field="target_class")
sk_time = get_sklearn_time_series_split_indices(train, k=K, date_field="created")

dist_kfold = compare_train_distributions(train, custom_kfold, sk_kfold, comparison_features)
dist_group = compare_train_distributions(train, custom_group_kfold, sk_group, comparison_features)
dist_strat = compare_train_distributions(train, custom_stratified_kfold, sk_strat, comparison_features)
dist_time = compare_train_distributions(train, custom_time_series, sk_time, comparison_features)

summary_dist = pd.DataFrame({
    "scheme": ["KFold", "GroupKFold", "StratifiedKFold", "TimeSeriesSplit"],
    "mean_abs_mean_diff": [
        dist_kfold["abs_mean_diff"].mean(),
        dist_group["abs_mean_diff"].mean(),
        dist_strat["abs_mean_diff"].mean(),
        dist_time["abs_mean_diff"].mean(),
    ],
    "mean_abs_std_diff": [
        dist_kfold["abs_std_diff"].mean(),
        dist_group["abs_std_diff"].mean(),
        dist_strat["abs_std_diff"].mean(),
        dist_time["abs_std_diff"].mean(),
    ],
})
display(summary_dist)
display(dist_strat.head(20))

,scheme,mean_abs_mean_diff,mean_abs_std_diff
0,KFold,5.167584e-16,2.648760e-13
1,GroupKFold,2.046436e+00,2.209269e+01
2,StratifiedKFold,5.281684e+00,5.764712e+02
3,TimeSeriesSplit,3.190586e-03,2.934166e-03


,fold,feature,custom_mean,sklearn_mean,abs_mean_diff,custom_std,sklearn_std,abs_std_diff
0,1,bathrooms,1.212267,1.212051,0.000215,0.501589,0.502103,0.000514
1,1,bedrooms,1.541704,1.541830,0.000127,1.114558,1.114757,0.000200
2,1,price,3841.718852,3730.227426,111.491426,24045.391831,8282.766943,15762.624888
3,1,latitude,40.742086,40.742161,0.000075,0.618767,0.618869,0.000102
4,1,longitude,-73.957149,-73.957220,0.000072,1.148163,1.148212,0.000049
5,1,num_photos,5.602239,5.607026,0.004787,3.606821,3.639985,0.033164
6,1,num_features,5.426408,5.423495,0.002913,3.926653,3.928474,0.001821
7,1,description_word_count,87.677440,87.538892,0.138548,58.607606,58.369242,0.238363
8,1,Elevator,0.533826,0.533978,0.000152,0.498861,0.498850,0.000010
9,1,DogsAllowed,0.445227,0.444062,0.001165,0.496997,0.496867,0.000130


## 9. Validation Scheme Comparison

Model: `StandardScaler + LinearRegression`.


In [13]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

def evaluate_folds(model, X, y, folds, name):
    rows = []
    for fold_id, (train_idx, test_idx) in enumerate(folds, start=1):
        X_train, y_train = X.loc[train_idx], y.loc[train_idx]
        X_test, y_test = X.loc[test_idx], y.loc[test_idx]
        result = fit_evaluate_model(model, X_train, y_train, X_test, y_test, model_name=name)
        result["fold"] = fold_id
        rows.append(result)
    return pd.DataFrame(rows)

cv_results = pd.concat([
    evaluate_folds(baseline_model, X, y, custom_kfold, "Custom KFold"),
    evaluate_folds(baseline_model, X, y, custom_group_kfold, "Custom GroupKFold"),
    evaluate_folds(baseline_model, X, y, custom_stratified_kfold, "Custom StratifiedKFold"),
    evaluate_folds(baseline_model, X, y, custom_time_series, "Custom TimeSeriesSplit"),
], ignore_index=True)

cv_summary = (
    cv_results
    .groupby("model")
    .agg(
        folds=("fold", "count"),
        mean_valid_RMSE=("valid_RMSE", "mean"),
        std_valid_RMSE=("valid_RMSE", "std"),
        mean_valid_MAE=("valid_MAE", "mean"),
        mean_valid_R2=("valid_R2", "mean"),
        mean_overfit_gap_RMSE=("overfit_gap_RMSE", "mean"),
        mean_fit_time_sec=("fit_time_sec", "mean"),
    )
    .sort_values("mean_valid_RMSE")
)

display(cv_results)
display(cv_summary)

,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE,fold
0,Custom KFold,0.116284,0.484050,0.597978,0.090342,0.484425,0.598996,0.073350,0.001017,1
1,Custom KFold,0.096848,0.483517,0.596943,0.092227,0.485164,0.601193,0.071470,0.004250,2
2,Custom KFold,0.097336,0.481860,0.596184,0.090575,0.487314,0.602539,0.083398,0.006355,3
3,Custom KFold,0.093458,0.485363,0.598667,0.088374,0.480156,0.592506,0.092723,-0.006161,4
4,Custom KFold,0.092560,0.483856,0.596718,0.086820,0.485159,0.600601,0.097792,0.003883,5
5,Custom GroupKFold,0.092344,0.484370,0.597964,0.093677,0.485912,0.596477,0.066259,-0.001487,1
6,Custom GroupKFold,0.102253,0.484452,0.598108,0.092266,0.485980,0.780793,-0.591682,0.182685,2
7,Custom GroupKFold,0.097970,0.482338,0.597473,0.092747,0.493180,0.598455,0.071407,0.000982,3
8,Custom GroupKFold,0.096407,0.493513,0.604090,0.078927,0.456751,0.572416,0.124418,-0.031674,4
9,Custom GroupKFold,0.091851,0.471282,0.586952,0.095931,0.519339,0.639590,0.053405,0.052638,5


,folds,mean_valid_RMSE,std_valid_RMSE,mean_valid_MAE,mean_valid_R2,mean_overfit_gap_RMSE,mean_fit_time_sec
model,,,,,,,
Custom StratifiedKFold,5,0.598862,0.004138,0.484409,0.084873,0.001553,0.095613
Custom KFold,5,0.599167,0.003935,0.484444,0.083747,0.001869,0.099297
Custom GroupKFold,5,0.637546,0.083637,0.488232,-0.055239,0.040629,0.096165
Custom TimeSeriesSplit,4,7.059588,12.077279,4.196528,-402.410164,6.466503,0.064088



For this dataset, the safest general-purpose choice is **Stratified K-Fold** on the preprocessed target.

Reason:

- The target classes are imbalanced: `low`, `medium`, `high`.
- Stratification keeps approximately the same target distribution in each fold.
- The dataset has a date field, but it is not a true forecasting task where production will necessarily predict only future months.
- GroupKFold by `manager_id` is also important if we want to test generalization to unseen managers, but it changes the business question.
- TimeSeriesSplit is best when the production process is strictly future prediction.


## 10. Time-Based 60/20/20 Split for Feature Selection

In [14]:
train_fs_df, valid_fs_df, test_fs_df = date_train_valid_test_split_by_ratio(
    train,
    date_field="created",
    train_size=0.6,
    validation_size=0.2,
    test_size=0.2,
)

X_train_fs = X.loc[train_fs_df.index]
y_train_fs = y.loc[train_fs_df.index]

X_valid_fs = X.loc[valid_fs_df.index]
y_valid_fs = y.loc[valid_fs_df.index]

X_test_fs = X.loc[test_fs_df.index]
y_test_fs = y.loc[test_fs_df.index]

print(X_train_fs.shape, X_valid_fs.shape, X_test_fs.shape)
print(train_fs_df["created"].min(), train_fs_df["created"].max())
print(valid_fs_df["created"].min(), valid_fs_df["created"].max())
print(test_fs_df["created"].min(), test_fs_df["created"].max())

(29611, 38) (9870, 38) (9871, 38)
2016-04-01 22:12:41 2016-05-25 01:10:42
2016-05-25 01:11:44 2016-06-12 08:17:43
2016-06-12 08:18:50 2016-06-29 21:41:47


## 11. Feature Selection Method 1 - Lasso coefficients

We fit Lasso with normalized features, sort features by absolute weight coefficient, then refit a model using the top 10 features.

In [15]:
lasso_full = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=5000)),
])

lasso_full.fit(X_train_fs, y_train_fs)

lasso_model = lasso_full.named_steps["model"]
lasso_coef = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "coef": lasso_model.coef_,
    "abs_coef": np.abs(lasso_model.coef_),
}).sort_values("abs_coef", ascending=False)

top10_lasso = lasso_coef.head(10)["feature"].tolist()

display(lasso_coef.head(20))
print("Top 10 Lasso features:", top10_lasso)

lasso_top10_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=5000)),
])

result_lasso_top10 = fit_evaluate_model(
    lasso_top10_model,
    X_train_fs[top10_lasso],
    y_train_fs,
    X_valid_fs[top10_lasso],
    y_valid_fs,
    model_name="Lasso top 10 coefficients",
)

result_lasso_top10_test = fit_evaluate_model(
    lasso_top10_model,
    pd.concat([X_train_fs[top10_lasso], X_valid_fs[top10_lasso]]),
    pd.concat([y_train_fs, y_valid_fs]),
    X_test_fs[top10_lasso],
    y_test_fs,
    model_name="Lasso top 10 coefficients final test",
)

display(pd.DataFrame([result_lasso_top10, result_lasso_top10_test]))

,feature,coef,abs_coef
9,created_hour,0.104990,0.104990
22,Doorman,-0.075593,0.075593
24,NoFee,0.064444,0.064444
0,bathrooms,-0.064283,0.064283
19,HardwoodFloors,0.062422,0.062422
26,FitnessCenter,-0.039832,0.039832
27,Pre-War,-0.031766,0.031766
25,LaundryinBuilding,0.027767,0.027767
32,HighSpeedInternet,0.024747,0.024747
1,bedrooms,0.022776,0.022776


Top 10 Lasso features: ['created_hour', 'Doorman', 'NoFee', 'bathrooms', 'HardwoodFloors', 'FitnessCenter', 'Pre-War', 'LaundryinBuilding', 'HighSpeedInternet', 'bedrooms']


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE
0,Lasso top 10 coefficients,0.109613,0.489068,0.601690,0.089398,0.482420,0.596627,0.096001,-0.005064
1,Lasso top 10 coefficients final test,0.116820,0.486633,0.600363,0.091277,0.497361,0.600399,0.031321,0.000036


## 12. Feature Selection Method 2 - nan-ratio and correlation

This method:

1. removes features with high missing-value ratio;
2. computes absolute Pearson correlation with the target;
3. returns the top 10 features.



In [16]:
def select_by_nan_ratio_and_correlation(
    X_raw: pd.DataFrame,
    y: pd.Series,
    max_nan_ratio: float = 0.3,
    top_n: int = 10,
):
    nan_ratio = X_raw.isna().mean()
    allowed_features = nan_ratio[nan_ratio <= max_nan_ratio].index.tolist()

    correlations = []
    for col in allowed_features:
        x_col = X_raw[col].replace([np.inf, -np.inf], np.nan)
        temp = pd.concat([x_col, y], axis=1).dropna()
        if temp[col].nunique() <= 1:
            corr = 0.0
        else:
            corr = abs(temp[col].corr(temp[y.name]))
        correlations.append({
            "feature": col,
            "nan_ratio": nan_ratio[col],
            "abs_corr": corr,
        })

    result = pd.DataFrame(correlations).sort_values("abs_corr", ascending=False)
    return result.head(top_n)["feature"].tolist(), result

X_raw_for_selection = train[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
top10_corr, corr_table = select_by_nan_ratio_and_correlation(
    X_raw_for_selection.loc[train_fs_df.index],
    y_train_fs.rename("target"),
    max_nan_ratio=0.3,
    top_n=10,
)

display(corr_table.head(20))
print("Top 10 nan-ratio/correlation features:", top10_corr)

corr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

result_corr_top10 = fit_evaluate_model(
    corr_model,
    X_train_fs[top10_corr],
    y_train_fs,
    X_valid_fs[top10_corr],
    y_valid_fs,
    model_name="Nan-ratio + Pearson corr top 10",
)

display(pd.DataFrame([result_corr_top10]))

,feature,nan_ratio,abs_corr
9,created_hour,0.0,0.178853
24,NoFee,0.0,0.128100
19,HardwoodFloors,0.0,0.117163
22,Doorman,0.0,0.092024
0,bathrooms,0.0,0.081057
35,LaundryInBuilding,0.0,0.075168
25,LaundryinBuilding,0.0,0.075168
27,Pre-War,0.0,0.069114
21,DogsAllowed,0.0,0.065817
30,OutdoorSpace,0.0,0.061569


Top 10 nan-ratio/correlation features: ['created_hour', 'NoFee', 'HardwoodFloors', 'Doorman', 'bathrooms', 'LaundryInBuilding', 'LaundryinBuilding', 'Pre-War', 'DogsAllowed', 'OutdoorSpace']


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE
0,Nan-ratio + Pearson corr top 10,0.021917,0.489558,0.602474,0.087023,0.483598,0.598447,0.090477,-0.004028


### 12.1 Chi-square 


In [17]:
X_train_non_negative = MinMaxScaler().fit_transform(X_train_fs)
chi2_scores, chi2_p_values = chi2(X_train_non_negative, y_train_fs)

chi2_table = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "chi2_score": chi2_scores,
    "p_value": chi2_p_values,
}).sort_values("chi2_score", ascending=False)

display(chi2_table.head(20))

,feature,chi2_score,p_value
24,NoFee,390.297804,1.769737e-85
19,HardwoodFloors,335.320890,1.534595e-73
35,LaundryInBuilding,189.651246,6.572872e-42
25,LaundryinBuilding,189.651246,6.572872e-42
22,Doorman,156.454756,1.062384e-34
23,Dishwasher,147.093381,1.145719e-32
9,created_hour,145.970551,2.008619e-32
30,OutdoorSpace,136.672297,2.098881e-30
27,Pre-War,111.748734,5.420893e-25
32,HighSpeedInternet,84.671951,4.108842e-19


## 13. Feature Selection Method 3 - permutation importance

We fit a baseline model and compute permutation importance on the validation sample.  
Then we select top 10 features and refit the model.

In [18]:
perm_base_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=5000)),
])

perm_base_model.fit(X_train_fs, y_train_fs)

start = time.time()
perm_result = permutation_importance(
    perm_base_model,
    X_valid_fs,
    y_valid_fs,
    n_repeats=3,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
)
perm_time = time.time() - start

perm_table = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False)

top10_perm = perm_table.head(10)["feature"].tolist()

display(perm_table.head(20))
print("Permutation importance time:", round(perm_time, 2), "sec")
print("Top 10 permutation features:", top10_perm)

perm_top10_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=5000)),
])

result_perm_top10 = fit_evaluate_model(
    perm_top10_model,
    X_train_fs[top10_perm],
    y_train_fs,
    X_valid_fs[top10_perm],
    y_valid_fs,
    model_name="Permutation importance top 10",
)

display(pd.DataFrame([result_perm_top10]))

,feature,importance_mean,importance_std
9,created_hour,0.018230,0.000002
22,Doorman,0.007756,0.000159
24,NoFee,0.007672,0.000432
19,HardwoodFloors,0.006952,0.000282
0,bathrooms,0.006047,0.000091
12,description_length,0.004126,0.000322
1,bedrooms,0.003340,0.000078
26,FitnessCenter,0.002125,0.000379
27,Pre-War,0.001789,0.000254
21,DogsAllowed,0.001447,0.000100


Permutation importance time: 0.89 sec
Top 10 permutation features: ['created_hour', 'Doorman', 'NoFee', 'HardwoodFloors', 'bathrooms', 'description_length', 'bedrooms', 'FitnessCenter', 'Pre-War', 'DogsAllowed']


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE
0,Permutation importance top 10,0.072473,0.489835,0.602667,0.086439,0.482752,0.596984,0.094918,-0.005683


## 14. Feature Selection Method 4 - SHAP

This section imports SHAP and uses it for feature ranking.


In [19]:
try:
    import shap
    SHAP_AVAILABLE = True
except Exception as e:
    SHAP_AVAILABLE = False
    print("SHAP is not available:", repr(e))
    print("If needed, install it with: pip install shap")

if SHAP_AVAILABLE:
    shap_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=5000)),
    ])
    shap_model.fit(X_train_fs, y_train_fs)

    scaler = shap_model.named_steps["scaler"]
    linear_model = shap_model.named_steps["model"]

    X_train_scaled = scaler.transform(X_train_fs)
    X_valid_scaled = scaler.transform(X_valid_fs)

    # Use sample for speed.
    sample_size = min(1000, X_valid_scaled.shape[0])
    rng = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(X_valid_scaled.shape[0], size=sample_size, replace=False)

    background_size = min(1000, X_train_scaled.shape[0])
    background_idx = rng.choice(X_train_scaled.shape[0], size=background_size, replace=False)
    explainer = shap.LinearExplainer(linear_model, X_train_scaled[background_idx])
    shap_values = explainer.shap_values(X_valid_scaled[sample_idx])

    shap_importance = np.abs(shap_values).mean(axis=0)
    shap_table = pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "mean_abs_shap": shap_importance,
    }).sort_values("mean_abs_shap", ascending=False)

    top10_shap = shap_table.head(10)["feature"].tolist()
    display(shap_table.head(20))
    print("Top 10 SHAP features:", top10_shap)

    shap_top10_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=5000)),
    ])

    result_shap_top10 = fit_evaluate_model(
        shap_top10_model,
        X_train_fs[top10_shap],
        y_train_fs,
        X_valid_fs[top10_shap],
        y_valid_fs,
        model_name="SHAP top 10",
    )

    display(pd.DataFrame([result_shap_top10]))
else:
    top10_shap = []
    result_shap_top10 = None

Background dataset has 1000 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1000 when initializing the masker.


,feature,mean_abs_shap
22,Doorman,0.075377
19,HardwoodFloors,0.063531
9,created_hour,0.062257
24,NoFee,0.060229
0,bathrooms,0.048986
26,FitnessCenter,0.035101
12,description_length,0.034238
1,bedrooms,0.030585
21,DogsAllowed,0.028001
27,Pre-War,0.027353


Top 10 SHAP features: ['Doorman', 'HardwoodFloors', 'created_hour', 'NoFee', 'bathrooms', 'FitnessCenter', 'description_length', 'bedrooms', 'DogsAllowed', 'Pre-War']


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE
0,SHAP top 10,0.07999,0.489835,0.602667,0.086439,0.482752,0.596984,0.094918,-0.005683


## 15. Compare feature selection methods


In [20]:
feature_selection_rows = []

feature_selection_rows.append({
    "method": "Lasso coefficients",
    "features": top10_lasso,
    "speed": "fast",
    "stability": "medium/high for linear models; depends on scaling and alpha",
    **{k: v for k, v in result_lasso_top10.items() if k.startswith("valid_") or k == "fit_time_sec"},
})

feature_selection_rows.append({
    "method": "Nan-ratio + Pearson corr",
    "features": top10_corr,
    "speed": "very fast",
    "stability": "high, but detects mostly linear single-feature relationships",
    **{k: v for k, v in result_corr_top10.items() if k.startswith("valid_") or k == "fit_time_sec"},
})

feature_selection_rows.append({
    "method": "Permutation importance",
    "features": top10_perm,
    "speed": "medium/slow",
    "stability": "medium; depends on validation split and repeats",
    **{k: v for k, v in result_perm_top10.items() if k.startswith("valid_") or k == "fit_time_sec"},
})

if result_shap_top10 is not None:
    feature_selection_rows.append({
        "method": "SHAP",
        "features": top10_shap,
        "speed": "medium/slow",
        "stability": "medium/high; strong interpretation method but depends on model",
        **{k: v for k, v in result_shap_top10.items() if k.startswith("valid_") or k == "fit_time_sec"},
    })

feature_selection_comparison = pd.DataFrame(feature_selection_rows).sort_values("valid_RMSE")
display(feature_selection_comparison)

,method,features,speed,stability,fit_time_sec,valid_MAE,valid_RMSE,valid_R2
0,Lasso coefficients,"[created_hour, Doorman, NoFee, bathrooms, Hard...",fast,medium/high for linear models; depends on scal...,0.109613,0.482420,0.596627,0.096001
3,SHAP,"[Doorman, HardwoodFloors, created_hour, NoFee,...",medium/slow,medium/high; strong interpretation method but ...,0.079990,0.482752,0.596984,0.094918
2,Permutation importance,"[created_hour, Doorman, NoFee, HardwoodFloors,...",medium/slow,medium; depends on validation split and repeats,0.072473,0.482752,0.596984,0.094918
1,Nan-ratio + Pearson corr,"[created_hour, NoFee, HardwoodFloors, Doorman,...",very fast,"high, but detects mostly linear single-feature...",0.021917,0.483598,0.598447,0.090477


## 16. Hyperparameter Optimization: Grid Search and Random Search

We optimize `alpha` and `l1_ratio` for `sklearn.linear_model.ElasticNet`.

In [21]:
def custom_grid_search_elasticnet(
    X_train,
    y_train,
    X_valid,
    y_valid,
    alpha_values,
    l1_ratio_values,
):
    rows = []
    best = None

    for alpha, l1_ratio in itertools.product(alpha_values, l1_ratio_values):
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                random_state=RANDOM_STATE,
                max_iter=5000,
            )),
        ])

        result = fit_evaluate_model(
            model,
            X_train,
            y_train,
            X_valid,
            y_valid,
            model_name=f"ElasticNet alpha={alpha}, l1_ratio={l1_ratio}",
        )
        result["alpha"] = alpha
        result["l1_ratio"] = l1_ratio
        rows.append(result)

        if best is None or result["valid_RMSE"] < best["valid_RMSE"]:
            best = result

    return pd.DataFrame(rows).sort_values("valid_RMSE"), best

def custom_random_search_elasticnet(
    X_train,
    y_train,
    X_valid,
    y_valid,
    alpha_range=(1e-5, 1.0),
    l1_ratio_range=(0.05, 1.0),
    n_iter=20,
    random_state=42,
):
    rng = np.random.RandomState(random_state)
    rows = []
    best = None

    for i in range(n_iter):
        # log-uniform alpha sampling
        log_alpha = rng.uniform(np.log10(alpha_range[0]), np.log10(alpha_range[1]))
        alpha = 10 ** log_alpha
        l1_ratio = rng.uniform(l1_ratio_range[0], l1_ratio_range[1])

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                random_state=RANDOM_STATE,
                max_iter=5000,
            )),
        ])

        result = fit_evaluate_model(
            model,
            X_train,
            y_train,
            X_valid,
            y_valid,
            model_name=f"ElasticNet random trial {i}",
        )
        result["alpha"] = alpha
        result["l1_ratio"] = l1_ratio
        result["trial"] = i
        rows.append(result)

        if best is None or result["valid_RMSE"] < best["valid_RMSE"]:
            best = result

    return pd.DataFrame(rows).sort_values("valid_RMSE"), best

### 16.1 Run custom Grid Search and Random Search

In [22]:
alpha_values = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
l1_ratio_values = [0.05, 0.25, 0.5, 0.75, 1.0]

grid_results, grid_best = custom_grid_search_elasticnet(
    X_train_fs, y_train_fs,
    X_valid_fs, y_valid_fs,
    alpha_values=alpha_values,
    l1_ratio_values=l1_ratio_values,
)

random_results, random_best = custom_random_search_elasticnet(
    X_train_fs, y_train_fs,
    X_valid_fs, y_valid_fs,
    alpha_range=(1e-5, 1.0),
    l1_ratio_range=(0.05, 1.0),
    n_iter=15,
    random_state=RANDOM_STATE,
)

print("Best Grid Search:")
display(pd.DataFrame([grid_best]))

print("Best Random Search:")
display(pd.DataFrame([random_best]))

display(grid_results.head(10))
display(random_results.head(10))

Best Grid Search:


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE,alpha,l1_ratio
0,"ElasticNet alpha=0.001, l1_ratio=0.25",10.748874,0.486301,0.599275,0.096696,0.48694,0.594462,0.102549,-0.004813,0.001,0.25


Best Random Search:


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE,alpha,l1_ratio,trial
0,ElasticNet random trial 8,17.542037,0.486174,0.599215,0.096876,0.486868,0.594437,0.102625,-0.004778,0.000332,0.548519,8


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE,alpha,l1_ratio
11,"ElasticNet alpha=0.001, l1_ratio=0.25",10.748874,0.486301,0.599275,0.096696,0.486940,0.594462,0.102549,-0.004813,0.00100,0.25
9,"ElasticNet alpha=0.0001, l1_ratio=1.0",23.932598,0.486077,0.599183,0.096972,0.486855,0.594488,0.102470,-0.004695,0.00010,1.00
10,"ElasticNet alpha=0.001, l1_ratio=0.05",20.096109,0.486117,0.599200,0.096919,0.486960,0.594488,0.102470,-0.004712,0.00100,0.05
8,"ElasticNet alpha=0.0001, l1_ratio=0.75",27.353880,0.486055,0.599178,0.096986,0.486863,0.594510,0.102402,-0.004668,0.00010,0.75
7,"ElasticNet alpha=0.0001, l1_ratio=0.5",27.821944,0.486034,0.599175,0.096997,0.486871,0.594535,0.102328,-0.004640,0.00010,0.50
6,"ElasticNet alpha=0.0001, l1_ratio=0.25",58.413079,0.486014,0.599172,0.097004,0.486880,0.594561,0.102251,-0.004612,0.00010,0.25
5,"ElasticNet alpha=0.0001, l1_ratio=0.05",76.631802,0.485997,0.599171,0.097007,0.486989,0.594589,0.102166,-0.004583,0.00010,0.05
4,"ElasticNet alpha=1e-05, l1_ratio=1.0",44.768210,0.485992,0.599171,0.097008,0.486975,0.594597,0.102141,-0.004574,0.00001,1.00
3,"ElasticNet alpha=1e-05, l1_ratio=0.75",40.049746,0.485990,0.599171,0.097008,0.486978,0.594600,0.102132,-0.004571,0.00001,0.75
2,"ElasticNet alpha=1e-05, l1_ratio=0.5",52.071298,0.485988,0.599171,0.097008,0.486980,0.594603,0.102122,-0.004568,0.00001,0.50


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE,alpha,l1_ratio,trial
8,ElasticNet random trial 8,17.542037,0.486174,0.599215,0.096876,0.486868,0.594437,0.102625,-0.004778,0.000332,0.548519,8
11,ElasticNet random trial 11,19.329391,0.486110,0.599192,0.096944,0.486871,0.594466,0.102537,-0.004726,0.000289,0.398044,11
13,ElasticNet random trial 13,22.701680,0.486037,0.599175,0.096996,0.486870,0.594531,0.102339,-0.004644,0.000100,0.538523,13
7,ElasticNet random trial 7,66.961616,0.486006,0.599172,0.097005,0.486880,0.594571,0.102220,-0.004601,0.000081,0.224234,7
3,ElasticNet random trial 3,26.487228,0.485999,0.599171,0.097007,0.486874,0.594582,0.102186,-0.004589,0.000020,0.872867,3
2,ElasticNet random trial 2,69.664561,0.485999,0.599171,0.097007,0.486973,0.594586,0.102174,-0.004585,0.000060,0.198195,2
5,ElasticNet random trial 5,73.921271,0.485995,0.599171,0.097008,0.486974,0.594594,0.102151,-0.004577,0.000013,0.971414,5
9,ElasticNet random trial 9,4.582805,0.486587,0.599416,0.096269,0.487134,0.594630,0.102043,-0.004786,0.001445,0.326668,9
0,ElasticNet random trial 0,2.261925,0.486820,0.599569,0.095808,0.487242,0.594839,0.101409,-0.004729,0.000746,0.953179,0
14,ElasticNet random trial 14,2.120947,0.487245,0.599704,0.095401,0.487533,0.595008,0.100901,-0.004696,0.009164,0.094128,14


### 16.2 Fit resulting best model and estimate test metrics

In [23]:
def fit_best_elasticnet_and_test(best_params, X_train, y_train, X_valid, y_valid, X_test, y_test, name):
    alpha = best_params["alpha"]
    l1_ratio = best_params["l1_ratio"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            random_state=RANDOM_STATE,
            max_iter=5000,
        )),
    ])

    X_train_valid = pd.concat([X_train, X_valid])
    y_train_valid = pd.concat([y_train, y_valid])

    result = fit_evaluate_model(
        model,
        X_train_valid,
        y_train_valid,
        X_test,
        y_test,
        model_name=name,
    )
    result["alpha"] = alpha
    result["l1_ratio"] = l1_ratio
    return result, model.fit(X_train_valid, y_train_valid)

grid_test_result, grid_final_model = fit_best_elasticnet_and_test(
    grid_best,
    X_train_fs,
    y_train_fs,
    X_valid_fs,
    y_valid_fs,
    X_test_fs,
    y_test_fs,
    "Best Grid Search ElasticNet",
)

random_test_result, random_final_model = fit_best_elasticnet_and_test(
    random_best,
    X_train_fs,
    y_train_fs,
    X_valid_fs,
    y_valid_fs,
    X_test_fs,
    y_test_fs,
    "Best Random Search ElasticNet",
)

display(pd.DataFrame([grid_test_result, random_test_result]))

,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE,alpha,l1_ratio
0,Best Grid Search ElasticNet,5.565713,0.483884,0.597863,0.098831,0.495115,0.719978,-0.392958,0.122115,0.001000,0.250000
1,Best Random Search ElasticNet,6.055262,0.483762,0.597831,0.098926,0.495615,0.747244,-0.500460,0.149412,0.000332,0.548519


## 17. Optuna Bayesian Optimization


In [24]:
try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception as e:
    OPTUNA_AVAILABLE = False
    print("Optuna is not available:", repr(e))
    print("If needed, install it with: pip install optuna")

if OPTUNA_AVAILABLE:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        alpha = trial.suggest_float("alpha", 1e-5, 1.0, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                random_state=RANDOM_STATE,
                max_iter=5000,
            )),
        ])

        model.fit(X_train_fs, y_train_fs)
        pred_valid = model.predict(X_valid_fs)
        rmse = rmse_score(y_valid_fs, pred_valid)
        return rmse

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=15)

    print("Best Optuna params:", study.best_params)
    print("Best Optuna RMSE:", study.best_value)

    optuna_best = {
        "alpha": study.best_params["alpha"],
        "l1_ratio": study.best_params["l1_ratio"],
    }

    optuna_test_result, optuna_final_model = fit_best_elasticnet_and_test(
        optuna_best,
        X_train_fs,
        y_train_fs,
        X_valid_fs,
        y_valid_fs,
        X_test_fs,
        y_test_fs,
        "Best Optuna ElasticNet",
    )

    display(pd.DataFrame([optuna_test_result]))

    optuna_trials = study.trials_dataframe()
    display(optuna_trials.sort_values("value").head(10))
else:
    optuna_test_result = None

Best Optuna params: {'alpha': 0.0004145446383542305, 'l1_ratio': 0.3840687246274145}
Best Optuna RMSE: 0.5944423969039273


,model,fit_time_sec,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,overfit_gap_RMSE,alpha,l1_ratio
0,Best Optuna ElasticNet,6.197725,0.483749,0.597829,0.098933,0.495627,0.748171,-0.504188,0.150342,0.000415,0.384069


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_l1_ratio,state
12,12,0.594442,2026-07-13 01:00:13.200641,2026-07-13 01:00:26.274807,0 days 00:00:13.074166,0.000415,0.384069,COMPLETE
9,9,0.594450,2026-07-13 00:59:29.850769,2026-07-13 00:59:46.464078,0 days 00:00:16.613309,0.000487,0.278521,COMPLETE
11,11,0.594450,2026-07-13 00:59:59.747684,2026-07-13 01:00:13.200606,0 days 00:00:13.452922,0.000353,0.392019,COMPLETE
10,10,0.594464,2026-07-13 00:59:46.464114,2026-07-13 00:59:59.747642,0 days 00:00:13.283528,0.000283,0.415827,COMPLETE
14,14,0.594493,2026-07-13 01:01:27.199191,2026-07-13 01:01:33.907663,0 days 00:00:06.708472,0.000680,0.540585,COMPLETE
3,3,0.594504,2026-07-13 00:57:12.352008,2026-07-13 00:57:41.790835,0 days 00:00:29.438827,0.001403,0.009977,COMPLETE
4,4,0.594517,2026-07-13 00:57:41.790886,2026-07-13 00:58:07.864153,0 days 00:00:26.073267,0.000103,0.664250,COMPLETE
0,0,0.594577,2026-07-13 00:56:41.851882,2026-07-13 00:57:10.816691,0 days 00:00:28.964809,0.000045,0.398484,COMPLETE
7,7,0.594606,2026-07-13 00:58:08.125024,2026-07-13 00:59:29.527843,0 days 00:01:21.402819,0.000035,0.066248,COMPLETE
13,13,0.594607,2026-07-13 01:00:26.274841,2026-07-13 01:01:27.199134,0 days 00:01:00.924293,0.000011,0.249167,COMPLETE


## 18. Optuna with Cross-Validation

In [25]:
if OPTUNA_AVAILABLE:
    cv_folds_for_optuna = custom_stratified_kfold[:5]

    def objective_cv(trial):
        alpha = trial.suggest_float("alpha", 1e-5, 1.0, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

        fold_rmses = []
        for train_idx, valid_idx in cv_folds_for_optuna:
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("model", ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1_ratio,
                    random_state=RANDOM_STATE,
                    max_iter=5000,
                )),
            ])

            model.fit(X.loc[train_idx], y.loc[train_idx])
            pred = model.predict(X.loc[valid_idx])
            rmse = rmse_score(y.loc[valid_idx], pred)
            fold_rmses.append(rmse)

        return float(np.mean(fold_rmses))

    study_cv = optuna.create_study(direction="minimize")
    study_cv.optimize(objective_cv, n_trials=8)

    print("Best Optuna CV params:", study_cv.best_params)
    print("Best Optuna CV mean RMSE:", study_cv.best_value)

    optuna_cv_results = study_cv.trials_dataframe().sort_values("value")
    display(optuna_cv_results.head(10))
else:
    print("Optuna CV skipped because Optuna is not available.")

Best Optuna CV params: {'alpha': 1.1017743281363727e-05, 'l1_ratio': 0.16928713036344567}
Best Optuna CV mean RMSE: 0.5988722594563474


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_l1_ratio,state
2,2,0.598872,2026-07-13 01:01:47.678586,2026-07-13 01:06:28.757717,0 days 00:04:41.079131,0.000011,0.169287,COMPLETE
4,4,0.598882,2026-07-13 01:06:29.070750,2026-07-13 20:26:10.365286,0 days 19:19:41.294536,0.000021,0.858560,COMPLETE
6,6,0.599138,2026-07-13 20:26:10.774180,2026-07-13 20:26:16.121302,0 days 00:00:05.347122,0.027378,0.029431,COMPLETE
1,1,0.610286,2026-07-13 01:01:47.099122,2026-07-13 01:01:47.678548,0 days 00:00:00.579426,0.092657,0.434779,COMPLETE
3,3,0.626029,2026-07-13 01:06:28.757782,2026-07-13 01:06:29.070713,0 days 00:00:00.312931,0.577141,0.633820,COMPLETE
0,0,0.626029,2026-07-13 01:01:46.556480,2026-07-13 01:01:47.099086,0 days 00:00:00.542606,0.886272,0.541165,COMPLETE
5,5,0.626029,2026-07-13 20:26:10.367036,2026-07-13 20:26:10.774146,0 days 00:00:00.407110,0.710723,0.517519,COMPLETE
7,7,0.626029,2026-07-13 20:26:16.121337,2026-07-13 20:26:16.533334,0 days 00:00:00.411997,0.661405,0.237360,COMPLETE


## 19. Hyperparameter Optimization Comparison

In [26]:
optimization_rows = [
    {
        "approach": "Custom Grid Search",
        "best_alpha": grid_best["alpha"],
        "best_l1_ratio": grid_best["l1_ratio"],
        "validation_RMSE": grid_best["valid_RMSE"],
        "test_RMSE": grid_test_result["valid_RMSE"],
        "test_MAE": grid_test_result["valid_MAE"],
        "comments": "Deterministic but checks all combinations and can be slow.",
    },
    {
        "approach": "Custom Random Search",
        "best_alpha": random_best["alpha"],
        "best_l1_ratio": random_best["l1_ratio"],
        "validation_RMSE": random_best["valid_RMSE"],
        "test_RMSE": random_test_result["valid_RMSE"],
        "test_MAE": random_test_result["valid_MAE"],
        "comments": "Faster for large spaces, but depends on random seed.",
    },
]

if optuna_test_result is not None:
    optimization_rows.append({
        "approach": "Optuna Bayesian Optimization",
        "best_alpha": optuna_best["alpha"],
        "best_l1_ratio": optuna_best["l1_ratio"],
        "validation_RMSE": study.best_value,
        "test_RMSE": optuna_test_result["valid_RMSE"],
        "test_MAE": optuna_test_result["valid_MAE"],
        "comments": "Learns from previous trials; usually efficient for expensive search.",
    })

optimization_comparison = pd.DataFrame(optimization_rows).sort_values("test_RMSE")
display(optimization_comparison)

,approach,best_alpha,best_l1_ratio,validation_RMSE,test_RMSE,test_MAE,comments
0,Custom Grid Search,0.001000,0.250000,0.594462,0.719978,0.495115,Deterministic but checks all combinations and ...
1,Custom Random Search,0.000332,0.548519,0.594437,0.747244,0.495615,"Faster for large spaces, but depends on random..."
2,Optuna Bayesian Optimization,0.000415,0.384069,0.594442,0.748171,0.495627,Learns from previous trials; usually efficient...


In [27]:
display(cv_summary)

,folds,mean_valid_RMSE,std_valid_RMSE,mean_valid_MAE,mean_valid_R2,mean_overfit_gap_RMSE,mean_fit_time_sec
model,,,,,,,
Custom StratifiedKFold,5,0.598862,0.004138,0.484409,0.084873,0.001553,0.095613
Custom KFold,5,0.599167,0.003935,0.484444,0.083747,0.001869,0.099297
Custom GroupKFold,5,0.637546,0.083637,0.488232,-0.055239,0.040629,0.096165
Custom TimeSeriesSplit,4,7.059588,12.077279,4.196528,-402.410164,6.466503,0.064088


For the current dataset, all hyperparameter optimization techniques gave very similar validation quality. Grid Search, Random Search and Optuna Bayesian Optimization obtained almost the same validation RMSE. However, Grid Search achieved the best final test RMSE and test MAE. Therefore, for this experiment, Grid Search is the best choice. It is deterministic, easy to reproduce and checks all specified hyperparameter combinations. Random Search is faster for large spaces but depends on the random seed. Optuna is usually more efficient for expensive experiments because it learns from previous trials, but here it did not improve the final test quality.